In [ ]:
import xarray as xr
import os   
import parflow as pf
import plotly.express as px
from parflow.tools.hydrology import calculate_overland_flow_grid, calculate_subsurface_storage, calculate_water_table_depth
import numpy as np
import shutil
import json
import plotly.io as pio

In [14]:
# import config from paper_figures/ (parent of this notebook's folder)
import sys
from pathlib import Path

try:
    _paper_figures = Path(__file__).resolve().parent.parent
except NameError:
    # Jupyter: __file__ is undefined; cwd is usually paper_figures or pumping/
    _cwd = Path.cwd()
    _paper_figures = _cwd.parent if _cwd.name == "pumping" else _cwd

sys.path.insert(0, str(_paper_figures))
import utils

In [15]:
ensemble_name = "baseline"
ensemble_member = "baseline"
baseline = utils.read_simulation_data(ensemble_name, ensemble_member, utils.DOMAIN)

/glade/derecho/scratch/bwest/drought-ensemble/analysis/paper_figures/utils.py:34: FutureWarning:

In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.



In [ ]:
data = utils.read_simulation_data(ensemble_name, ensemble_member, utils.DOMAIN_2)
data.info()

In [ ]:
# total_storage = data.storage.sel(time=slice(None, None, 8760)).sum(dim=["x", "y","z"])
# total_storage_baseline = baseline.storage.sel(time=slice(None, None, 8760)).sum(dim=["x", "y","z"])

In [ ]:
px.line(data.out)

In [20]:
import plotly.graph_objects as go

# Compute yearly PME total from the same domain used by this baseline run
raw_nc_path = Path(json.loads(Path(
    f"/glade/derecho/scratch/bwest/drought-ensemble/domains/{utils.DOMAIN}/processed_full_runs/{ensemble_name}/{ensemble_member}/file_locations.json"
).read_text())[0])
run_dir = raw_nc_path.parent
pme = pf.read_pfb(str(run_dir / "pme.pfb"))

if pme.ndim != 3:
    raise ValueError(f"Unexpected pme.pfb shape: {pme.shape}")

# Domain mask (active cells only)
mask_2d = baseline.mask.isel(time=0, z=0).values > 0

# pme.pfb can be stored with a single active z layer; collapse if needed
layer_activity = np.sum(np.abs(pme), axis=(1, 2))
active_layers = np.where(layer_activity > 0)[0]
if active_layers.size == 1:
    pme_2d = pme[active_layers[0], :, :]
else:
    pme_2d = np.sum(pme, axis=0)

# Use the same grid spacing as the yearly PME notebook.
# Processed NetCDF x/y are index coordinates (0,1,2,...) not physical meters.
DX = 1000.0
DY = 1000.0
HOURS_PER_YEAR = 365.0 * 24.0

yearly_pme_total_m3 = float(
    np.sum(np.where(mask_2d, pme_2d, 0.0)) * DX * DY * HOURS_PER_YEAR
)
if yearly_pme_total_m3 == 0:
    raise ValueError("Domain-integrated yearly PME is zero; cannot compute fraction.")

# Year-to-year storage change (finite-difference derivative over each 1-year interval)
years = total_storage_baseline.time.values
delta_storage_m3 = np.diff(total_storage_baseline.values)
interval_mid_years = 0.5 * (years[:-1] + years[1:])
fraction_of_yearly_pme = delta_storage_m3 / yearly_pme_total_m3

graph = go.Figure()
graph.add_trace(go.Scatter(
    x=interval_mid_years,
    y=fraction_of_yearly_pme,
    mode='lines+markers',
    name='Baseline annual ΔS / yearly PME'
))
graph.add_hline(y=0.0, line_width=1, line_color='black')
graph.update_layout(
    xaxis_title="year",
    yaxis_title="Storage change / yearly PME (-)",
    title="Annual storage change as fraction of yearly PME"
)

graph.show()

# Consistency check: this should be 1.0 for uniform 1-year spacing.
dt_years = np.diff(years)
backward_derivative_m3_per_year = delta_storage_m3 / dt_years
print(f"Using yearly PME total = {yearly_pme_total_m3:,.3e} m^3/year")
print(
    f"Fraction range: {np.nanmin(fraction_of_yearly_pme):.4f} to "
    f"{np.nanmax(fraction_of_yearly_pme):.4f}"
)
print(
    "corr(ΔS, dS/dt over interval) = "
    f"{np.corrcoef(delta_storage_m3, backward_derivative_m3_per_year)[0, 1]:.6f}"
)

# Extra sanity checks for "by-eye" slope comparisons on the first graph.
start_end_single_year_ratio = abs(delta_storage_m3[-1]) / abs(delta_storage_m3[0])
start_end_five_year_ratio = (
    abs(np.mean(delta_storage_m3[-5:])) / abs(np.mean(delta_storage_m3[:5]))
)
print(
    "|ΔS_last_year| / |ΔS_first_year| = "
    f"{start_end_single_year_ratio:.3f}"
)
print(
    "|mean(ΔS last 5y)| / |mean(ΔS first 5y)| = "
    f"{start_end_five_year_ratio:.3f}"
)


Using yearly PME total = 1.523e+09 m^3/year
Fraction range: -0.0075 to -0.0033
corr(ΔS, dS/dt over interval) = 1.000000
|ΔS_last_year| / |ΔS_first_year| = 0.436
|mean(ΔS last 5y)| / |mean(ΔS first 5y)| = 0.529
